# 04 - Real data: the Hillstrom e-mail experiment

The MineThatData E-Mail Analytics dataset (Kevin Hillstrom, 2008): 64,000 customers randomly assigned to receive a men's merchandise e-mail, a women's merchandise e-mail, or no e-mail, with two weeks of follow-up (`visit`, `conversion`, `spend`). Each customer also carries `history`: actual dollars spent in the prior year, our pre-experiment CUPED covariate.

**DESIGN DECISION: two-arm restriction.** We compare **Womens E-Mail (treatment) vs No E-Mail (control)** and drop the Mens E-Mail arm. The decision engine models a single treatment-vs-control contrast; a three-arm comparison would need its own multiplicity treatment and muddy the demonstration.

In [1]:
import sys
sys.path.insert(0, "..")  # so config and src import from the repo root

import os
import urllib.request

import numpy as np
import pandas as pd

from src.cuped import compute_theta, cuped_adjust
from src.decision_engine import MetricSpec, evaluate_experiment
from src.tests_stats import difference_in_means

URL = ("http://www.minethatdata.com/"
       "Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv")
PATH = "../data/hillstrom.csv"
if not os.path.exists(PATH):
    urllib.request.urlretrieve(URL, PATH)  # cached in data/, gitignored

raw = pd.read_csv(PATH)
df = raw[raw["segment"].isin(["No E-Mail", "Womens E-Mail"])].copy()
df["group"] = np.where(df["segment"] == "Womens E-Mail", "treatment", "control")

df.groupby("group")[["visit", "conversion", "spend"]].agg(["mean", "size"])

visit        conversion            spend       
               mean   size       mean   size      mean   size
group                                                        
control    0.106167  21306   0.005726  21306  0.652789  21306
treatment  0.151400  21387   0.008837  21387  1.077202  21387

## 1. Metric mapping

| role | metric | test |
|---|---|---|
| primary success | `conversion` (bought within 2 weeks) | one-sided superiority, BH family |
| secondary success | `visit` (site visit within 2 weeks) | one-sided superiority, BH family |
| guardrail | `spend` per user, ALL users | non-inferiority vs 10% of control mean |

**The guardrail is spend per ALL users (zeros included), not spend per converter.** Per-converter ratio metrics condition on a post-treatment variable: if the e-mail changes WHO converts, the two converter pools are no longer comparable and the comparison picks up composition (selection) bias, not a treatment effect. We hit exactly this in our synthetic generator: a single latent factor driving both conversion and order value made a pure conversion lift mechanically depress treatment AOV by ~2%, swallowing the entire margin (see `project_log.md`, incident 3, and the two-latent redesign that fixed it). All-user spend is randomization-protected: its denominator (all randomized users) cannot be moved by treatment.

**DESIGN DECISION: non-inferiority margin = 10% of control mean spend.** Business framing: a merchandising e-mail must not cannibalize short-term revenue; finance accepts at most a 10% dip in revenue per user over the campaign window as the cost of longer-term engagement. In a real company this number is set by finance/leadership BEFORE the experiment, not by the analyst after seeing the data. In dollars: control mean spend is $0.65 per user, so the margin is about $0.065 per user.

In [2]:
specs = [
    MetricSpec(name="conversion rate", kind="success", column="conversion",
               test="proportion", is_primary=True),
    MetricSpec(name="visit rate", kind="success", column="visit", test="proportion"),
    MetricSpec(name="spend per user (all users)", kind="guardrail", column="spend",
               test="mean", direction="decrease", margin=0.10, margin_type="relative"),
]
report = evaluate_experiment(df, specs)
print(report.render())

# Decision report

## VERDICT: INVESTIGATE

- primary improved, but at least one guardrail is inconclusive

alpha = 0.05, guardrail power floor = 80%

### conversion rate (success, proportion test)
- effect: +0.00311 (one-sided 95% lower bound +0.00176)
- one-sided superiority p = 0.0001; significant after BH: YES
- status: PASS

### visit rate (success, proportion test)
- effect: +0.04523 (one-sided 95% lower bound +0.03991)
- one-sided superiority p = 0.0000; significant after BH: YES
- status: PASS

### spend per user (all users) (guardrail, mean test)
- effect: +0.42441, margin: -0.06528 (relative 0.1), worst-case bound: +0.21003
- non-inferiority p = 0.0001 (established); power to clear margin if harmless = 13%
- status: INCONCLUSIVE  (non-inferior but power 13% below floor)



## 2. Reading the verdict

Both success metrics improve decisively (conversion +0.31pp on a 0.57% base, visit +4.5pp; both BH-significant). The spend guardrail actually CLEARS its margin: the e-mail RAISES spend, and even the worst-case bound sits above -$0.065. The engine still says INVESTIGATE because the guardrail's a priori power is far below the 80% floor: at ~21k users per arm, with spend zero for ~99% of users and a heavy-tailed positive tail (control SD ~$11.6 vs a $0.65 mean), a $0.065 margin was never realistically testable. The pass we observed is driven by a large positive effect, not by an adequately sized design.

This is an honest and common real-world outcome: the engine is telling us the guardrail DESIGN is underpowered, and a pass under low power deserves scrutiny (had the true effect been zero, the test would have been inconclusive ~87% of the time). A real team would either accept the observed bound as sufficient evidence, widen the margin, or run longer.

## 3. Real-data CUPED: spend adjusted by prior-year history

On the synthetic data, CUPED cut revenue variance by ~37% because the pre-period covariate correlated at rho ~0.61 with the outcome. Expectations here must be set honestly: `spend` is zero for ~99% of users in a two-week window, so its correlation with ANY pre-period covariate is structurally capped near zero, and `history` (a full prior year of spend) is a coarse signal for a two-week outcome. A real but very small reduction is the expected result, and that is what we find.

In [3]:
y = df["spend"].values.astype(float)
x = df["history"].values.astype(float)
is_t = (df["group"] == "treatment").values

rho = np.corrcoef(y, x)[0, 1]
y_adj = cuped_adjust(y, x)
reduction = 1 - np.var(y_adj, ddof=1) / np.var(y, ddof=1)

raw = difference_in_means(y[is_t], y[~is_t])
adj = difference_in_means(y_adj[is_t], y_adj[~is_t])
narrowing = 1 - (adj.ci_high - adj.ci_low) / (raw.ci_high - raw.ci_low)

print(f"theta = {compute_theta(y, x):.6f},  rho(spend, history) = {rho:.4f}")
print(f"variance reduction: {reduction:.3%}  (theoretical rho^2 = {rho**2:.3%})")
print(f"raw spend effect:   {raw.estimate:+.4f}  [{raw.ci_low:+.4f}, {raw.ci_high:+.4f}]")
print(f"CUPED spend effect: {adj.estimate:+.4f}  [{adj.ci_low:+.4f}, {adj.ci_high:+.4f}]")
print(f"CI narrowing: {narrowing:.3%}")
print(f"\nContrast with synthetic: rho 0.61 -> 37% variance reduction there;")
print(f"rho {rho:.3f} -> {reduction:.2%} here. Same method, same math; the gain")
print("is entirely a property of how predictive the covariate is.")

theta = 0.001035,  rho(spend, history) = 0.0195
variance reduction: 0.038%  (theoretical rho^2 = 0.038%)
raw spend effect:   +0.4244  [+0.1690, +0.6799]
CUPED spend effect: +0.4227  [+0.1673, +0.6781]
CI narrowing: 0.019%

Contrast with synthetic: rho 0.61 -> 37% variance reduction there;
rho 0.020 -> 0.04% here. Same method, same math; the gain
is entirely a property of how predictive the covariate is.


## 4. What I would request in a real company

- **Per-user pre-period panels**: spend, visits and orders in the 4-8 weeks immediately before the experiment, per user. `history` spans a whole year; a recent, same-cadence window would correlate far better with the outcome.
- **Richer covariates**: recency, frequency, channel and category affinity; even the `history_segment`, `recency` and `zip_code` fields shipped with this dataset could be combined into a stronger covariate.
- **CUPAC as the natural extension**: instead of one covariate, train a model that predicts the outcome from ALL pre-experiment features and use its prediction as the CUPED covariate (Doordash's CUPAC). CUPED with theta on a single x is the special case; a good prediction pushes rho, and the variance reduction, much higher.
- **A margin set by the business before launch**, with the guardrail's power at the planned sample size reviewed at design time, so an underpowered guardrail is caught before the experiment runs, not after.

## Takeaways

- The engine transfers to real data unchanged: same specs, same tests, same report.
- Success metrics can be overwhelming while a guardrail remains formally inconclusive; INVESTIGATE is a feature, not a failure: it surfaces that the guardrail design (margin x sample size x metric variance) was never adequate.
- CUPED's value is bounded by covariate quality. Dramatic on synthetic revenue (rho 0.61), nearly nil on two-week zero-inflated spend vs year-long history (rho ~0.02). The method is identical; the data decides.
- Guardrails belong on all-user metrics whenever possible; per-converter ratios invite post-treatment selection bias.